In this notebook, we perform the all the computations presented in the note. We first trained the helper model. We then train the student model with a pretrained helper model and a randomly initialized helper model. We then plot the loss curves with number of token batches. We also sample the outputs of the three trained models.

In [1]:
from google.colab import drive
import sys
import os
import json
from datasets import load_dataset
folder_path = '/content/drive/MyDrive/Colab Notebooks'
if folder_path not in sys.path:
    sys.path.append(folder_path)
drive.mount('/content/drive', force_remount=True)

code_dir = "/content/drive/MyDrive/Colab Notebooks"
tokenizer = "/content/drive/MyDrive/Colab Notebooks/cosmopedia-tokenizer-20k"
output_dir = "/content/drive/MyDrive/Colab Notebooks/checkpoints"


# We use these training parameters for training all three models
epochs = 1 # this is the number of times our learning schedule resets. The total number of iterations = epochs*iterations_per_epoch
iterations_per_epoch = 10
batch_size = 20
max_seq_len = 256
check_every = 10


# these are parameters for the helper model
helper_d_model = 512
helper_num_heads = 8
helper_num_layers = 6
helper_window = 15
helper_theta = 15000

# these are parameters for the student model
student_d_model = 1024
student_num_heads = 8
student_num_layers = 8
student_theta = 10000
stride = 8

# these are parameters for the cross-attention mechanism
cross_num_heads = 8
num_cross_blocks = 6
cross_scale = 1.0
cross_theta = 25000


Mounted at /content/drive


In [ ]:
''' We train the helper model with the Cosmopedia dataset. '''

!python "$code_dir/pretrain_helper.py" \
--code-dir "$code_dir" \
--tokenizer "$tokenizer" \
--output-dir "$output_dir" \
--epochs $epochs \
--iterations-per-epoch $iterations_per_epoch \
--batch-size $batch_size \
--max-seq-len $max_seq_len \
--helper-d-model $helper_d_model \
--helper-num-heads $helper_num_heads \
--helper-num-layers $helper_num_layers \
--helper-window $helper_window \
--helper-theta $helper_theta\
--check-every $check_every


In [ ]:
''' We train the student model with the pretrained helper model. We use the FineWebEdu dataset.'''

!python "$code_dir/train_pretrained_helper.py" \
  --code-dir "$code_dir" \
  --tokenizer "$tokenizer" \
  --helper-checkpoint "$output_dir/checkpointPhraseFinal.pt" \
  --output-dir "$output_dir" \
  --epochs $epochs \
  --iterations-per-epoch $iterations_per_epoch \
  --batch-size $batch_size \
  --max-seq-len $max_seq_len \
  --helper-d-model $helper_d_model \
  --student-d-model $student_d_model \
  --helper-num-heads $helper_num_heads \
  --student-num-heads $student_num_heads \
  --cross-num-heads $cross_num_heads \
  --helper-num-layers $helper_num_layers \
  --student-num-layers $student_num_layers \
  --student-theta $student_theta\
  --cross-theta $cross_theta\
  --helper-window $helper_window \
  --stride $stride \
  --num-cross-blocks $num_cross_blocks \
  --cross-scale $cross_scale\
  --check-every $check_every


In [ ]:
''' We train the student model with the a randomly initialized helper model.  We use the FineWebEdu dataset. '''

!python "$code_dir/train_random_helper.py" \
--code-dir "$code_dir" \
--tokenizer "$tokenizer" \
--output-dir "$output_dir" \
--epochs $epochs \
--iterations-per-epoch $iterations_per_epoch \
--batch-size $batch_size \
--max-seq-len $max_seq_len \
--helper-d-model $helper_d_model \
--student-d-model $student_d_model \
--helper-num-heads $helper_num_heads \
--student-num-heads $student_num_heads \
--cross-num-heads $cross_num_heads \
--helper-num-layers $helper_num_layers \
--student-num-layers $student_num_layers \
--helper-window $helper_window \
--stride $stride \
--num-cross-blocks $num_cross_blocks \
--cross-scale $cross_scale\
--check-every $check_every

In [ ]:
'''In this cell, we compare the training loss trajectories'''

# here, stride = 1 for pooled average computation
window_size = 30

import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks')
from reproduce_figure_3 import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random_loss = smooth_loss(load_loss_history(os.path.join(output_dir, "checkpointRandomHelperFinal.pt"), device), window_size = window_size)
pretrained_loss = smooth_loss(load_loss_history(os.path.join(output_dir, "checkpointPretrainedHelperFinal.pt"), device), window_size = window_size)
plt.figure(figsize=(7, 4.5))
plt.plot(random_loss, label="Random helper")
plt.plot(pretrained_loss, label="Pretrained helper")
plt.xlabel("Training batch")
plt.ylabel("Mean next-token loss")
plt.legend()
plt.show()

In [ ]:
'''In this cell, we upload our models and inspect their outputs'''

from common_functionsv13 import NewTransformerLM, PhraseTransformerLM
import sys
import torch
from transformers import AutoTokenizer
from tokenizers import decoders


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/cosmopedia-tokenizer-20k",
    local_files_only=True,
)
tokenizer.backend_tokenizer.decoder = decoders.ByteLevel()

checkpoint_paths =['/content/drive/MyDrive/Colab Notebooks/checkpoints/checkpointRandomHelperFinal.pt',
                   '/content/drive/MyDrive/Colab Notebooks/checkpoints/checkpointPretrainedHelperFinal.pt',
                   '/content/drive/MyDrive/Colab Notebooks/checkpoints/checkpointPhraseFinal.pt']

checkpoint = [torch.load(checkpoint_paths[i], map_location=device, weights_only=False) for i in range(3)]

model_pretrained = NewTransformerLM(
    tokenizer.vocab_size,
    helper_d_model,    # helper d_model
    student_d_model,   # student d_model
    helper_num_heads,      # helper heads
    student_num_heads,      # student heads
    cross_num_heads,      # cross-attention heads
    helper_num_layers,      # helper layers
    student_num_layers,      # student layers
    student_theta,  # student theta
    helper_theta,  # helper theta
    cross_theta,  # cross-attention theta
    max_seq_len,    # max sequence length
    helper_window,     # helper window
    stride,      # dilation stride
    num_cross_blocks,      # cross-attention blocks
    device=device,
).to(device)
model_untrained = NewTransformerLM(
    tokenizer.vocab_size,
    helper_d_model,    # helper d_model
    student_d_model,   # student d_model
    helper_num_heads,      # helper heads
    student_num_heads,      # student heads
    cross_num_heads,      # cross-attention heads
    helper_num_layers,      # helper layers
    student_num_layers,      # student layers
    student_theta,  # student theta
    helper_theta,  # helper theta
    cross_theta,  # cross-attention theta
    max_seq_len,    # max sequence length
    helper_window,     # helper window
    stride,      # dilation stride
    num_cross_blocks,      # cross-attention blocks
    device=device,
).to(device)
model_phrase = PhraseTransformerLM(
    tokenizer.vocab_size,
    helper_d_model,
    helper_num_heads,
    helper_num_layers,
    helper_theta,
    helper_window,
    device=device
).to(device)


model_untrained.load_state_dict((checkpoint[0])["model"])
model_untrained.eval()
model_pretrained.load_state_dict((checkpoint[1])["model"])
model_pretrained.eval()
model_phrase.load_state_dict((checkpoint[2])["model"])
model_phrase.eval()
print("The models have now been uploaded.")


In [ ]:
# @title
'''In this cell, we print out outputs of the three models, i.e., helper model, student model + untrained helper,
student model + pretrained mode, in this order.
'''

from common_functionsv13 import decode_old, decode_new
text = 'I ate a sandwich'



text_out = decode_old(tokenizer, text, model_phrase, 20, device)
print(text_out)
text_out = decode_new(tokenizer, text, 1, model_untrained, 20, device)
print(text_out)
text_out = decode_new(tokenizer, text, 1, model_pretrained, 20, device)
print(text_out)
